# Purpose 
## This notebook looks at the keys, what each variable means, structure of the dataset and identifies what's actually usable for the analysis

The notebook reads:
- `scrape_log.csv` to identify successful requests.
- JSON files from `raw_match_centres/`.

### Getting successful records of JSON from all downloaded JSONS

In [63]:
Path.cwd()

PosixPath('/Users/ananyachaba/Desktop/LedeProgram/Projects/Football/data_work/analysis')

In [64]:
data_collection = Path("../data_collection") #.. means go to your parent folder, which is .../Football/data_work. and then go to data collection/
print(data_collection.resolve()) #this is how you get absolute path i.e. where you exactly are!

/Users/ananyachaba/Desktop/LedeProgram/Projects/Football/data_work/data_collection


In [ ]:
from pathlib import Path
import pandas as pd
import json

# read project folder (1 folder up from where we are right now)
data_collection = Path("../data_collection")

# read scrape log
log = pd.read_csv(data_collection / "scrape_log.csv")

#keeping only successful logs
log_success = log[log["status"] == "success"]

# getting file NAMES from the successful logs in a list [] called success_files
success_files = log_success["file_name"].tolist()

# we have the file names of success files. Now we want the actual files, not just the names. For that, we need to define the path for python.
json_folder = data_collection / "raw_match_centres" # note that raw_match_centre is where all your downloaded JSONs are from your scraper

# getting success files and saving them in a list called records
records = []

for file in success_files: #looping on each file in success_files 
    with open(json_folder / file, "r", encoding="utf-8") as f: #adding them here
        records.append(json.load(f))

# sanity check 
len(records)
# 21,027 success records out of ~23k scraped records.

### Exploring records

In [68]:
records[0]

{'fixture': {'id': 70600,
  'tournament_id': 2094,
  'date': '19 June 2026,',
  'time': '05:45 PM',
  'venue': '\nPoornima University',
  'stadium': None,
  'team1_id': 19665,
  'team2_id': 14583,
  'tournament_name': "R-LEAGUE YOUTH U13 (Boy's) 2025-26",
  'team1_logo': 'https://aiff.b-cdn.net/uploads/sm_IMG3531_1726942210.jpeg',
  'team2_logo': 'https://aiff.b-cdn.net/logos/sm_14583-logo_small-1629881153.png',
  't1_name': 'Rajasthan Football School',
  't2_name': 'Football Club Brothers United',
  'ft': True,
  'ft_text': 'This match has finished'},
 'teams_data': {'14583': {'short_name': None,
   'logo': 'logos/sm_14583-logo_small-1629881153.png',
   'name': 'Football Club Brothers United'},
  '19665': {'short_name': None,
   'logo': 'uploads/sm_IMG3531_1726942210.jpeg',
   'name': 'Rajasthan Football School'}},
 'tournament': {'id': 2094,
  'name': "R-LEAGUE YOUTH U13 (Boy's) 2025-26",
  'team_type': 2},
 'team1_players': [{'captain': 1,
   'goal_keeper': 1,
   'jersey_no': 1,
   

In [69]:
type(records[0])

dict

In [70]:
records[0].keys()

dict_keys(['fixture', 'teams_data', 'tournament', 'team1_players', 'team2_players', 'squad1_list', 'squad2_list', 'events', 'team1_score', 'team2_score', 'report', 'lineup1', 'lineup2', 'official1', 'official2', 'mvp', 'stats_available'])

#### Exploring key 1: fixture

In [71]:
records[0]['fixture'].keys()

dict_keys(['id', 'tournament_id', 'date', 'time', 'venue', 'stadium', 'team1_id', 'team2_id', 'tournament_name', 'team1_logo', 'team2_logo', 't1_name', 't2_name', 'ft', 'ft_text'])

In [72]:
records[0]['fixture']

{'id': 70600,
 'tournament_id': 2094,
 'date': '19 June 2026,',
 'time': '05:45 PM',
 'venue': '\nPoornima University',
 'stadium': None,
 'team1_id': 19665,
 'team2_id': 14583,
 'tournament_name': "R-LEAGUE YOUTH U13 (Boy's) 2025-26",
 'team1_logo': 'https://aiff.b-cdn.net/uploads/sm_IMG3531_1726942210.jpeg',
 'team2_logo': 'https://aiff.b-cdn.net/logos/sm_14583-logo_small-1629881153.png',
 't1_name': 'Rajasthan Football School',
 't2_name': 'Football Club Brothers United',
 'ft': True,
 'ft_text': 'This match has finished'}

##### Key: Fixture

| Field                      | Meaning                      | Likely use                                    |
| -------------------------- | ---------------------------- | --------------------------------------------- |
| `id`                       | Fixture/match ID             | this is what you use to get to a specific match, numeric ID of match centre used to scrape records                       
| `tournament_id`            | Tournament/competition ID                | Useful for grouping matches under one competition|
|`tournament_name`          | Tournament/competition label | **Must correspond to tournament_id 1:1**                |
| `date`                     | Match date                   | **Needs cleaning**            |
| `time`                     | Match time                   | Probably optional                             |
| `venue`                    | Venue name                   | Potentially useful; needs whitespace cleaning |
| `stadium`                  | Stadium field                | Check how often it is missing                 |
| `team1_id`, `team2_id`     | Team IDs                     | Essential                                     |
| `t1_name`, `t2_name`       | Team names                   | Essential                                     |
| `ft`                       | Whether match finished       | Important eligibility check, to confirm meaning                  |
| `ft_text`                  | Human-readable match status  | Useful for validating `ft`                    |
| `team1_logo`, `team2_logo` | Image URLs                   | Not needed for analysis                       |


#### Exploring key 2: teams data

In [73]:
records[0]["teams_data"]

{'14583': {'short_name': None,
  'logo': 'logos/sm_14583-logo_small-1629881153.png',
  'name': 'Football Club Brothers United'},
 '19665': {'short_name': None,
  'logo': 'uploads/sm_IMG3531_1726942210.jpeg',
  'name': 'Rajasthan Football School'}}

In [16]:
records[0]["teams_data"].keys()

dict_keys(['14583', '19665'])

**teams_data: dictionary keyed by team ID containing name of team, logo and short name of team. Appears to duplicate information already present in fixture. Not used for analysis.**

#### Exploring key 3: tournament

In [18]:
records[0]["tournament"]

{'id': 2094, 'name': "R-LEAGUE YOUTH U13 (Boy's) 2025-26", 'team_type': 2}

**tournament: contains tournament ID, name and team_type. ID and name duplicate fixture; ***retain team_type temporarily*** and test its unique values across all records before deciding whether it is useful.**

#### Exploring key 4: team1_players

In [23]:
type(records[0]["team1_players"])
# its a list

list

In [24]:
len(records[0]["team1_players"])
# team 1 has 29 players

29

In [25]:
type(records[0]["team1_players"][0])

dict

In [74]:
records[0]["team1_players"][0]

{'captain': 1,
 'goal_keeper': 1,
 'jersey_no': 1,
 'jersey_name': '',
 'playing_position': 2,
 'status': 1,
 'aiff_id': 418848,
 'player_name': 'Kabir Mehta',
 'player_id': 492465,
 'team_id': 19665}

In [27]:
# it is a list of dictionaries with one player = one dictionary



| Field              | Keep?        | Notes                                                                                                                         |
| ------------------ | ------------ | ----------------------------------------------------------------------------------------------------------------------------- |
| `captain`          | Yes (for now) | Flag indicating captain. Probably irrelevant to appearances but still keep.                                                              |
| `goal_keeper`      | No           | Position-specific; not relevant.                                                                                              |
| `jersey_no`        | No           | Not a stable identifier.                                                                                                      |
| `jersey_name`      | No           | Display name only.                                                                                                            |
| `playing_position` | No       | Keep temporarily until you inspect its unique values. Might be useful for future analyses, but not this story.                |
| `status`           | **Yes**      | **Critical.** Need to determine exactly what 1, 2, 3... mean because this determines whether a player made an appearance. |
| `aiff_id`          | **Yes**      | Very likely the canonical AIFF player ID. **Test uniqueness across matches.**                                                    |
| `player_name`      | **Yes**      | Useful for validation and reporting, but don't use it as the primary identifier.                                              |
| `player_id`        | **Yes**      | **Keep until you understand how it differs from `aiff_id`.** One may be stable and the other match-specific.                      |
| `team_id`          | **Yes**      | Useful for joining to match/team data and checking transfers.                                                               
Checked a couple of matches:
Status = 1 → Players on the pitch at kickoff (starting XI).
Status = 2 → Players on the bench, regardless of whether they eventually came on.
Status = 3 → Players in the squad who were neither in the starting XI nor on the matchday bench.

Confirmed after looking at type under events:

| Status | Interpretation                             |
| ------ | ------------------------------------------ |
| 1      | Named in the starting XI                   |
| 2      | Named on the matchday bench                |
| 3      | In the squad but not on the matchday bench |



#### Exploring key 5: team2_players

In [29]:
type(records[0]["team2_players"])
# its a list

list

In [32]:
len(records[0]["team2_players"])
# team 1 has 28 players
# teams can have different players

28

In [33]:
records[0]["team2_players"][0].keys()

dict_keys(['captain', 'goal_keeper', 'jersey_no', 'jersey_name', 'playing_position', 'status', 'aiff_id', 'player_name', 'player_id', 'team_id'])

In [34]:
# same as team1_players

#### Exploring key 6: squad1_list

In [36]:
type(records[0]["squad1_list"])

list

In [37]:
records[0]["squad1_list"]
# empty list

[]

In [38]:
records[1]["squad1_list"]

[]

In [39]:
records[2]["squad1_list"]

[]

In [40]:
records[3]["squad1_list"]

[]

In [41]:
records[10]["squad1_list"]

[]

In [42]:
any(len(r["squad1_list"]) > 0 for r in records)

True

In [43]:
for i, r in enumerate(records):
    if len(r["squad1_list"]) > 0:
        print(i)
        break

14


records[i]["squad1_list"]

**squad1_list: Usually empty. A small number of records (14) contain a simplified squad list (player_id, player_name, jersey_no, playing_position). As it lacks appearance-related fields and duplicates information available in team1_players, it was not used in the analysis.**

#### Exploring key 7: squad2_list

In [46]:
type(records[0]["squad2_list"])

list

In [47]:
records[0]["squad2_list"]
# empty list

[]

In [49]:
for i, r in enumerate(records):
    if len(r["squad2_list"]) > 0:
        print(i)
        break

14


In [50]:
records[i]["squad2_list"]

[{'player_id': 304533,
  'player_name': 'Om Ohol',
  'jersey_no': 2,
  'playing_position': 4},
 {'player_id': 304499,
  'player_name': 'Ranjeet Ramesh Joshi',
  'jersey_no': 3,
  'playing_position': 4},
 {'player_id': 341095,
  'player_name': 'Mohammad Kaif Kamruddin Sayyed',
  'jersey_no': 4,
  'playing_position': 4},
 {'player_id': 341094,
  'player_name': 'DANISH CHANDARGI',
  'jersey_no': 5,
  'playing_position': 4},
 {'player_id': 304504,
  'player_name': 'Vaibhav Sandesh Patil',
  'jersey_no': 8,
  'playing_position': 3},
 {'player_id': 304500,
  'player_name': 'Stallion Nitin Arland',
  'jersey_no': 9,
  'playing_position': 3},
 {'player_id': 304495,
  'player_name': 'Naeem Sayad',
  'jersey_no': 10,
  'playing_position': 1},
 {'player_id': 304506,
  'player_name': 'Waseem Sayad',
  'jersey_no': 12,
  'playing_position': 3},
 {'player_id': 304496,
  'player_name': 'NIKHIL NAIR',
  'jersey_no': 13,
  'playing_position': 4},
 {'player_id': 304488,
  'player_name': 'Akash Manoranja

squad2_list: Usually empty. A small number of records (14) contain a simplified squad list (player_id, player_name, jersey_no, playing_position). As it lacks appearance-related fields and duplicates information available in team1_players, it was not used in the analysis.

#### Exploring key 8: events

In [75]:
type(records[0]["events"])

list

In [76]:
records[0]["events"][0]

{'id': 682003,
 'aiff_player_id': 361306,
 'sub_aiff_player_id': 0,
 'minute': 61,
 'plus_time': 0,
 'bm_am': 0,
 'type': 1,
 'goal_type': 1,
 'second_yellow': 0,
 'team_id': 14583,
 'show_type': 'event',
 'ts': 6100,
 'player_name': 'Mohammed Jishan'}

| Field                | Keep? | Description                                                                                                                 |
| -------------------- | :---: | --------------------------------------------------------------------------------------------------------------------------- |
| `id`                 |   No  | Unique identifier for the event. Not used in analysis.                                                                      |
| `aiff_player_id`     |  Yes  | AIFF ID of the player associated with the event (e.g. goal scorer, player substituted off). Used to link events to players. |
| `sub_aiff_player_id` |  Yes  | AIFF ID of the player entering the match during a substitution. `0` for non-substitution events.                            |
| `minute`             | No | Match minute when the event occurred. Not required for appearance counts.     |
| `plus_time`          |   No  | Additional stoppage time associated with the event. Not used.                                                               |
| `bm_am`              |   No  | Unknown field. Appears to be `0` in inspected records. Not used.                                                            |
| `type`               |  Yes  | Event type. Verified values so far: `1 = Goal`, `4 = Substitution`.           |
| `goal_type`          |   No  | Type of goal (e.g. normal, penalty, own goal). Not required for appearance analysis.                                        |
| `second_yellow`      |   No  | Indicates whether a red card resulted from a second yellow. Not used.                                                       |
| `team_id`            | Yes | Team associated with the event. Retained if team-level event analysis is required.                                          |
| `show_type`          |   No  | Display field used by the website (e.g. `"event"`). Not used.                                                               |
| `ts`                 |   No  | Timestamp used for ordering/display (appears to encode match time). Not used.                                               |
| `player_name`        |   No  | Player name associated with the event. Useful for manual validation but not used as an identifier.                          |


Notes
* Substitution events (type == 4):
* aiff_player_id = player substituted off.
* sub_aiff_player_id = player substituted on.
* Goal events (type == 1):
* aiff_player_id = goal scorer.
* sub_aiff_player_id = 0.


| Key                          | Decision           | Why                                                                                                                                                                                 |
| ---------------------------- | ------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `team1_score`, `team2_score` | Ignore           | Not needed for counting appearances.                                                            |
| `report`                     | Ignore             | PDF paths only. Potentially useful for manual spot-checking, but not for the main dataset.                                                                                          |
| `lineup1`, `lineup2`         | Ignore             | Partial, position-grouped version of players already contained more completely in `team1_players` and `team2_players`.|
| `official1`, `official2`     | Ignore             | Appears to describe team officials/coaches, unrelated to player appearances.                                                                                                        |
| `mvp`                        | Ignore             | Not relevant.                                                                                                                                                                       |
| `stats_available`            | Ignore |     |


* confirm status and type - confirmed for now
* check stadium missing values
* check if tournament_id and tournament_name correspond 1:1 (one id can't have multiple names and vice versa)
* check dates format, filter for season
* check if team_type is always == 2
* figure out aiff_id and player_id differences
* clean competition names and categories